In [2]:
function* mergeSortCoroutine(arr) {
  if (arr.length <= 1) return arr

  function* helper(sub) {
    if (sub.length <= 1) return sub
    const mid = Math.floor(sub.length / 2)
    const left = yield* helper(sub.slice(0, mid))
    const right = yield* helper(sub.slice(mid))
    const merged = []
    let i = 0,
      j = 0
    while (i < left.length && j < right.length) {
      const comparison = yield [left[i], right[j]]
      if (comparison === null) {
        merged.push(left[i++])
        // merged.push(left[i])
        // i++
        // j++
      } else if (comparison) {
        merged.push(left[i])
        i++
      } else {
        merged.push(right[j])
        j++
      }
    }
    while (i < left.length) {
      merged.push(left[i])
      i++
    }
    while (j < right.length) {
      merged.push(right[j])
      j++
    }
    return merged
  }

  const sorted = yield* helper(arr)
  return sorted
}


In [3]:
function* timSortCoroutine(arr) {
  if (arr.length <= 1) return arr

  const MIN_RUN = 8

  function minRunLength(n) {
    let r = 0
    while (n >= MIN_RUN) {
      r |= n & 1
      n >>= 1
    }
    return n + r
  }

  function* compare(a, b) {
    const c = yield [a, b]
    return c
  }

  function* insertionSort(run) {
    for (let i = 1; i < run.length; i++) {
      const value = run[i]
      let j = i - 1

      while (j >= 0) {
        const cmp = yield* compare(run[j], value)
        if (cmp === null || cmp) break
        run[j + 1] = run[j]
        j--
      }

      run[j + 1] = value
    }

    return run
  }

  function* merge(left, right) {
    const merged = []
    let i = 0
    let j = 0

    while (i < left.length && j < right.length) {
      const cmp = yield* compare(left[i], right[j])

      if (cmp === null) {
        merged.push(left[i++])
        j++
      } else if (cmp) {
        merged.push(left[i++])
      } else {
        merged.push(right[j++])
      }
    }

    while (i < left.length) merged.push(left[i++])
    while (j < right.length) merged.push(right[j++])

    return merged
  }

  const minRun = minRunLength(arr.length)
  const runs = []

  let i = 0
  while (i < arr.length) {
    // Detect run
    let j = i + 1

    if (j < arr.length) {
      const asc = yield* compare(arr[i], arr[j])

      if (asc === null || asc) {
        // ascending
        while (j + 1 < arr.length) {
          const c = yield* compare(arr[j], arr[j + 1])
          if (!(c === null || c)) break
          j++
        }
      } else {
        // descending
        while (j + 1 < arr.length) {
          const c = yield* compare(arr[j], arr[j + 1])
          if (c === null || c) break
          j++
        }
      }
    }

    let run = arr.slice(i, j + 1)

    // Reverse descending run
    if (run.length > 1) {
      const c = yield* compare(run[0], run[1])
      if (!(c === null || c)) run.reverse()
    }

    // Extend to minimum run
    const end = Math.min(arr.length, i + minRun)
    if (run.length < minRun) {
      run = arr.slice(i, end)
      run = yield* insertionSort(run)
      i = end
    } else {
      i = j + 1
    }

    runs.push(run)
  }

  // Merge runs
  while (runs.length > 1) {
    const mergedRuns = []

    for (let k = 0; k < runs.length; k += 2) {
      if (k + 1 < runs.length)
        mergedRuns.push(yield* merge(runs[k], runs[k + 1]))
      else mergedRuns.push(runs[k])
    }

    runs.splice(0, runs.length, ...mergedRuns)
  }

  return runs[0]
}


In [4]:
function* modifiedTimSortCoroutine(arr) {
  if (arr.length <= 1) return arr

  function* compare(a, b) {
    return yield [a, b]
  }

  function* merge(left, right) {
    // Already in order?
    const boundary = yield* compare(left[left.length - 1], right[0])

    if (boundary === null || boundary) {
      return left.concat(right)
    }

    const merged = []
    let i = 0
    let j = 0

    while (i < left.length && j < right.length) {
      const c = yield* compare(left[i], right[j])

      if (c === null || c) {
        // left <= right
        merged.push(left[i++])
      } else {
        merged.push(right[j++])
      }
    }

    while (i < left.length) merged.push(left[i++])

    while (j < right.length) merged.push(right[j++])

    return merged
  }

  // -----------------------------
  // Detect natural runs
  // -----------------------------

  const runs = []
  let i = 0

  while (i < arr.length) {
    let end = i

    if (end + 1 < arr.length) {
      const firstCmp = yield* compare(arr[end], arr[end + 1])
      const ascending = firstCmp === null || firstCmp

      end++

      while (end < arr.length - 1) {
        const c = yield* compare(arr[end], arr[end + 1])

        if (ascending) {
          if (!(c === null || c)) break
        } else {
          if (c === null || c) break
        }

        end++
      }

      const run = arr.slice(i, end + 1)

      if (!ascending) run.reverse()

      runs.push(run)
    } else {
      runs.push([arr[i]])
    }

    i = end + 1
  }

  // -----------------------------
  // Merge runs
  // -----------------------------

  while (runs.length > 1) {
    // Find the adjacent pair with the smallest total length.
    let best = 0
    let bestSize = runs[0].length + runs[1].length

    for (let i = 1; i < runs.length - 1; i++) {
      const size = runs[i].length + runs[i + 1].length
      if (size < bestSize) {
        best = i
        bestSize = size
      }
    }

    const merged = yield* merge(runs[best], runs[best + 1])

    runs.splice(best, 2, merged)
  }

  return runs[0]
}


In [5]:
function runSort(sortFn, array) {
    const input = [...array];
    const gen = sortFn(input);

    let comparisons = 0;
    let result = gen.next();

    while (!result.done) {
        const [a, b] = result.value;

        comparisons++;

        let response;
        if (a < b)
            response = true;
        else if (a > b)
            response = false;
        else
            response = null;

        result = gen.next(response);
    }

    return {
        sorted: result.value,
        comparisons,
    };
}

function arraysEqual(a, b) {
    return (
        a.length === b.length &&
        a.every((v, i) => v === b[i])
    );
}

function isSorted(arr) {
    for (let i = 1; i < arr.length; i++)
        if (arr[i - 1] > arr[i])
            return false;
    return true;
}

function randomArray(length, max = 1000) {
    return Array.from(
        { length },
        () => Math.floor(Math.random() * max)
    );
}

function shuffledRange(n) {
    const arr = [...Array(n).keys()];
    for (let i = n - 1; i > 0; i--) {
        const j = Math.floor(Math.random() * (i + 1));
        [arr[i], arr[j]] = [arr[j], arr[i]];
    }
    return arr;
}

let tests = [

    {
        name: "Already sorted",
        data: [...Array(30).keys()]
    },

    {
        name: "Reverse sorted",
        data: [...Array(30).keys()].reverse()
    },

    {
        name: "One adjacent swap",
        data: (() => {
            const a = [...Array(30).keys()];
            [a[14], a[15]] = [a[15], a[14]];
            return a;
        })()
    },

    {
        name: "One item moved",
        data: (() => {
            const a = [...Array(30).keys()];
            const x = a.splice(5, 1)[0];
            a.splice(20, 0, x);
            return a;
        })()
    },

    {
        name: "Mostly sorted (5 random swaps)",
        data: (() => {
            const a = [...Array(100).keys()];
            for (let i = 0; i < 5; i++) {
                const x = Math.floor(Math.random() * a.length);
                const y = Math.floor(Math.random() * a.length);
                [a[x], a[y]] = [a[y], a[x]];
            }
            return a;
        })()
    },

    {
        name: "Mostly sorted (10 random swaps)",
        data: (() => {
            const a = [...Array(100).keys()];
            for (let i = 0; i < 10; i++) {
                const x = Math.floor(Math.random() * a.length);
                const y = Math.floor(Math.random() * a.length);
                [a[x], a[y]] = [a[y], a[x]];
            }
            return a;
        })()
    },

    {
        name: "Mostly sorted (30 random swaps)",
        data: (() => {
            const a = [...Array(100).keys()];
            for (let i = 0; i < 30; i++) {
                const x = Math.floor(Math.random() * a.length);
                const y = Math.floor(Math.random() * a.length);
                [a[x], a[y]] = [a[y], a[x]];
            }
            return a;
        })()
    },

    {
        name: "Random (30)",
        data: randomArray(30)
    },

    {
        name: "Random (100)",
        data: randomArray(100)
    },

    {
        name: "Random (300)",
        data: randomArray(300)
    },

    {
        name: "Many duplicates",
        data: Array.from({length:100}, () => Math.floor(Math.random()*10))
    },

    {
        name: "Random permutation",
        data: shuffledRange(100)
    },
];

for (const test of tests) {

    const expected = [...test.data].sort((a, b) => a - b);

    const merge = runSort(mergeSortCoroutine, test.data);
    const tim = runSort(timSortCoroutine, test.data);
    const modified = runSort(modifiedTimSortCoroutine, test.data);

    const mergeOK =
        isSorted(merge.sorted) &&
        arraysEqual(merge.sorted, expected);

    const timOK =
        isSorted(tim.sorted) &&
        arraysEqual(tim.sorted, expected);

    const modifiedOK =
        isSorted(modified.sorted) &&
        arraysEqual(modified.sorted, expected);

    console.group(test.name);

    console.log(
        "Merge:",
        merge.comparisons,
        "comparisons",
        mergeOK ? "✓" : "✗"
    );

    console.log(
        "Tim:",
        tim.comparisons,
        "comparisons",
        timOK ? "✓" : "✗"
    );

    console.log(
        "Modified:",
        modified.comparisons,
        "comparisons",
        modifiedOK ? "✓" : "✗"
    );

    // const diff = merge.comparisons - modified.comparisons;

    // if (diff > 0)
    // console.log(`TimSort saved ${diff} comparisons.`);
    // else if (diff < 0)
    //     console.log(`MergeSort saved ${-diff} comparisons.`);
    // else
    //     console.log("Tie.");

    // console.log('MergeSort took')

    console.groupEnd();
}

Already sorted
  Merge: 71 comparisons ✓
  Tim: 30 comparisons ✓
  Modified: 29 comparisons ✓
Reverse sorted
  Merge: 77 comparisons ✓
  Tim: 30 comparisons ✓
  Modified: 29 comparisons ✓
One adjacent swap
  Merge: 72 comparisons ✓
  Tim: 47 comparisons ✓
  Modified: 46 comparisons ✓
One item moved
  Merge: 74 comparisons ✓
  Tim: 52 comparisons ✓
  Modified: 51 comparisons ✓
Mostly sorted (5 random swaps)
  Merge: 445 comparisons ✓
  Tim: 458 comparisons ✓
  Modified: 434 comparisons ✓
Mostly sorted (10 random swaps)
  Merge: 451 comparisons ✓
  Tim: 458 comparisons ✓
  Modified: 450 comparisons ✓
Mostly sorted (30 random swaps)
  Merge: 518 comparisons ✓
  Tim: 597 comparisons ✓
  Modified: 562 comparisons ✓
Random (30)
  Merge: 113 comparisons ✓
  Tim: 141 comparisons ✓
  Modified: 130 comparisons ✓
Random (100)
  Merge: 531 comparisons ✓
  Tim: 625 comparisons ✗
  Modified: 601 comparisons ✓
Random (300)
  Merge: 2088 comparisons ✓
  Tim: 2256 comparisons ✗
  Modified: 2338 compari

In [ ]:
mergeTotal = 0;
timTotal = 0;
modifiedTotal = 0;

for (let i = 0; i < 1000; i++) {
    const arr = [...Array(100).keys()];

    // 5: modified wins, 6: merge wins
    for (let k = 0; k < 5; k++){
        // Introduce k random swaps.
        for (let j = 0; j < k; j++) {
            const a = Math.floor(Math.random() * arr.length);
            const b = Math.floor(Math.random() * arr.length);
            [arr[a], arr[b]] = [arr[b], arr[a]];
        }
    }

    mergeTotal += runSort(mergeSortCoroutine, arr).comparisons;
    timTotal += runSort(timSortCoroutine, arr).comparisons;
    modifiedTotal += runSort(modifiedTimSortCoroutine, arr).comparisons;
}

console.log("Average merge comparisons:", mergeTotal / 1000);
console.log("Average tim comparisons:  ", timTotal / 1000);
console.log("Average modified comparisons:  ", modifiedTotal / 1000);

Average merge comparisons: 460.67
Average tim comparisons:   496.269
Average modified comparisons:   443.462
